In [2]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
import pandas as pd

train_df = pd.read_csv('../dados/train_processado.csv')
test_df = pd.read_csv('../dados/test_processado.csv')

In [4]:
# separar as features (X) da coluna alvo/rótulo (y)
# tira 'Id_Passageiro' e 'Sobreviveu' das features usadas para prever
features = [col for col in train_df.columns if col not in ['Id_Passageiro', 'Sobreviveu', 'Irmaos_Conjuges', 'Pais_Filhos']]

X = train_df[features]
y = train_df['Sobreviveu']

In [5]:
# Dividir em conjunto de treino (80%) e validação (20%)
X_treino, X_val, y_treino, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# instanciar e treinar o modelo de Árvore de Decisão
modelo = DecisionTreeClassifier(max_depth=3, random_state=41)
modelo.fit(X_treino, y_treino)

# Fazer previsões nos dados de validação (que o modelo nunca viu antes)
previsoes = modelo.predict(X_val)

# Calcular a acurácia (porcentagem de acertos)
acuracia = accuracy_score(y_val, previsoes)

print(f"Acurácia da arvore de decisao na validação: {acuracia * 100:.2f}%")

Acurácia da arvore de decisao na validação: 79.89%


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Instanciando uma Floresta com 20 árvores
floresta = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=42)
floresta.fit(X_treino, y_treino)

previsoes_floresta = floresta.predict(X_val)
acuracia_floresta = accuracy_score(y_val, previsoes_floresta)

print(f"Acurácia da Random Forest na validação: {acuracia_floresta * 100:.2f}%")

Acurácia da Random Forest na validação: 84.92%


In [7]:
# Cria um DataFrame ordenado com a importância de cada feature
importancia_features = pd.DataFrame({
    'Caracteristica': features,
    'Importancia': floresta.feature_importances_
}).sort_values(by='Importancia', ascending=False)

display(importancia_features)

,Caracteristica,Importancia
1,Sexo,0.399275
0,ClassePassageiro,0.141531
3,ValorBilhete,0.131414
2,Idade,0.117287
4,Total_Familiar,0.056215
15,Deck_U,0.040410
7,Porto_Embarque_S,0.037940
11,Deck_E,0.021074
8,Deck_B,0.020902
5,Sozinho,0.011804


In [8]:
# Média de sobrevivência agrupada
taxa_sobrevivencia = train_df.groupby(['Sexo', 'ClassePassageiro'])['Sobreviveu'].mean() * 100
print(taxa_sobrevivencia)

Sexo  ClassePassageiro
0     1                   36.885246
      2                   15.740741
      3                   13.544669
1     1                   96.808511
      2                   92.105263
      3                   50.000000
Name: Sobreviveu, dtype: float64


In [9]:
# Média de sobrevivência agrupada
taxa_sobrevivencia = train_df.groupby(['Sexo', 'Idade'])['Sobreviveu'].mean() * 100
print(taxa_sobrevivencia)

Sexo  Idade
0     0.42     100.0
      0.67     100.0
      0.83     100.0
      0.92     100.0
      1.00      60.0
               ...  
1     57.00      0.0
      58.00    100.0
      60.00    100.0
      62.00    100.0
      63.00    100.0
Name: Sobreviveu, Length: 147, dtype: float64


In [14]:
import pandas as pd

# remove 'Id_Passageiro' e 'Sobreviveu' 
features_teste = [col for col in test_df.columns if col != 'Id_Passageiro' and col != 'Irmaos_Conjuges' and col != 'Pais_Filhos']
X_teste = test_df[features_teste]
X_teste = X_teste.reindex(columns=X.columns, fill_value=0)

#  Fazer as previsões com o modelo treinado
previsoes_teste = floresta.predict(X_teste)

# cria o DataFrame de submissão
submissao = pd.DataFrame({
    'PassengerId': test_df['Id_Passageiro'],
    'Survived': previsoes_teste
})

# salva
submissao.to_csv('../dados/resultado_submission.csv', index=False)